In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
# 1. 라이브러리
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

def rename_files_to_match_ids(audio_dir, csv_df):
    filenames = os.listdir(audio_dir)
    original_filenames = set(filenames)

    rename_count = 0
    for audio_id in csv_df['ID']:
        if not audio_id.endswith(".wav"):
            target_name = audio_id + ".wav"

        simplified_name = audio_id.replace('_I_', '_').replace('_E_', '_') + ".wav"

        if simplified_name in original_filenames:
            old_path = os.path.join(audio_dir, simplified_name)
            new_path = os.path.join(audio_dir, target_name)
            os.rename(old_path, new_path)
            rename_count += 1

    print(f"✅ {audio_dir} 내 파일 {rename_count}개 이름 변경 완료")
# 경로 설정
TRAIN_AUDIO_DIR = "/content/drive/MyDrive/ml_project/kaggle/train"
TEST_AUDIO_DIR = "/content/drive/MyDrive/ml_project/kaggle/test"

# CSV 로딩
train_df = pd.read_csv("/content/drive/MyDrive/ml_project/kaggle/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/ml_project/kaggle/test.csv")

# 파일명 정리 실행
rename_files_to_match_ids(TRAIN_AUDIO_DIR, train_df)
rename_files_to_match_ids(TEST_AUDIO_DIR, test_df)

# 2. 설정
SR = 16000
N_MELS = 64
DURATION = 3  # 초
BATCH_SIZE = 32
EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. 데이터셋
class AudioDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=True, label_encoder=None):
        self.df = df
        self.audio_dir = audio_dir
        self.is_train = is_train
        self.le = label_encoder
        if self.is_train and self.le is None:
            self.le = LabelEncoder()
            self.df['Target'] = self.le.fit_transform(df['Target'])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_id = row['ID']
        if not audio_id.endswith(".wav"):
            audio_id += ".wav"
        file_path = os.path.join(self.audio_dir, audio_id)

        if not os.path.exists(file_path):
            print(f"[경고] 파일 없음: {file_path}")
            logmel = torch.zeros((1, N_MELS, 94))
            if self.is_train:
                return logmel, torch.tensor(0)
            else:
                return logmel, row['ID']

        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS)
        logmel = librosa.power_to_db(mel)
        logmel = logmel[:, :SR * DURATION // 512]
        logmel = torch.tensor(logmel).unsqueeze(0).float()

        if self.is_train:
            label = torch.tensor(self.le.transform([row['Target']])[0]).long()
            return logmel, label
        else:
            return logmel, row['ID']
# 4. 모델 정의
class CNN_GRU_Model(nn.Module):
    def __init__(self, input_channels=1, rnn_hidden_size=64, rnn_layers=1, num_classes=2):
        super(CNN_GRU_Model, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 16, 3, 1, 1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.rnn_input_size = 32 * 16  # CNN 마지막 채널 × 주파수 방향 크기
        self.rnn = nn.GRU(input_size=self.rnn_input_size, hidden_size=rnn_hidden_size,
                          num_layers=rnn_layers, batch_first=True, bidirectional=True)

        self.fc = nn.Sequential(
            nn.Linear(rnn_hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)  # (B, C, F, T)
        B, C, F, T = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(B, T, -1)  # (B, T, C*F)
        x, _ = self.rnn(x)
        x = x[:, -1, :]  # 마지막 타임스텝의 출력
        return self.fc(x)
# 5. 학습/평가 함수
def train_model(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(DEVICE)
            output = model(x)
            pred = torch.argmax(output, axis=1).cpu().numpy()
            preds.extend(pred)
            trues.extend(y.numpy())
    return accuracy_score(trues, preds)
# 6. 학습 실행
train_df = pd.read_csv("/content/drive/MyDrive/ml_project/kaggle/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/ml_project/kaggle/test.csv")

label_encoder = LabelEncoder()
label_encoder.fit(train_df["Target"])

train_dataset = AudioDataset(train_df, TRAIN_AUDIO_DIR, is_train=True, label_encoder=label_encoder)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = CNN_GRU_Model().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    loss = train_model(model, train_loader, criterion, optimizer)
    acc = evaluate_model(model, train_loader)
    print(f"[{epoch+1}] Loss: {loss:.4f} | Train Acc: {acc:.4f}")
# 7. 테스트 예측
test_dataset = AudioDataset(test_df, TEST_AUDIO_DIR, is_train=False, label_encoder=label_encoder)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
ids, labels = [], []
with torch.no_grad():
    for x, id_list in test_loader:
        x = x.to(DEVICE)
        y_pred = model(x)
        pred_label = torch.argmax(y_pred, axis=1).cpu().numpy()
        ids.extend(id_list)
        labels.extend(label_encoder.inverse_transform(pred_label))

# 8. 제출 파일
submission = pd.DataFrame({'ID': ids, 'Target': labels})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv 생성 완료")